---
image: example.gif
execute: 
  enabled: true
---

# Highlighting different periods with a repeating overlay

In [ ]:
from feat_repeating_overlay_model_classes import Trial, g
from vidigi.prep import reshape_for_animations, generate_animation_df
from vidigi.animation import generate_animation, add_repeating_overlay
from vidigi.utils import EventPosition, create_event_position_df
import os
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "feat_repeating_overlay_model_classes.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
g.sim_duration = 60 * 24 * 7 # 7 days
g.number_of_runs = 1

my_trial = Trial()

my_trial.run_trial()

In [ ]:
my_trial.all_event_logs.head(50)

In [ ]:
# Create a list of EventPosition objects
event_position_df = create_event_position_df([
    EventPosition(event='arrival', x=50, y=300, label="Arrival"),
    EventPosition(event='treatment_wait_begins', x=205, y=275, label="Waiting for Treatment"),
    EventPosition(event='treatment_begins', x=205, y=175, resource='n_cubicles', label="Being Treated"),
    EventPosition(event='depart', x=270, y=70, label="Exit")
])

In [ ]:
my_trial.all_event_logs[my_trial.all_event_logs['run']==0]

In [ ]:
LIMIT_DURATION = g.sim_duration
WRAP_QUEUES_AT = 15
STEP_SNAPSHOT_MAX = WRAP_QUEUES_AT * 4

full_patient_df = reshape_for_animations(
    event_log=my_trial.all_event_logs[my_trial.all_event_logs['run']==0],
    every_x_time_units=10,
    entity_col_name="patient",
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    limit_duration=LIMIT_DURATION,
    debug_mode=True
    )

full_patient_df.head(15)

In [ ]:
full_patient_df_plus_pos = generate_animation_df(
    full_entity_df=full_patient_df,
    event_position_df=event_position_df,
    entity_col_name="patient",
    wrap_queues_at=WRAP_QUEUES_AT,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    gap_between_entities=10,
    gap_between_resources=10,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    debug_mode=True
    )

full_patient_df_plus_pos.sort_values(['patient', 'snapshot_time']).head(15)

In [ ]:
full_patient_df_plus_pos[full_patient_df_plus_pos["patient"]=="overnight_closure"]

In [ ]:
fig = generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['patient', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=g(),
        entity_col_name="patient",
        debug_mode=True,
        setup_mode=False,
        include_play_button=True,
        start_time="08:00:00",
        entity_icon_size=20,
        resource_icon_size=20,
        gap_between_resource_rows=30,
        plotly_height=700,
        frame_duration=800,
        frame_transition_duration=200,
        plotly_width=1200,
        override_x_max=300,
        override_y_max=500,
        time_display_units="day_clock_ampm",
        display_stage_labels=False,
        add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_1_simplest_case/Simplest%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
    )

fig

## Adding the overlay

If you drag the slider through to 8pm, you will see an overlay showing the clinic is closed. This will continue through until 8am the next day.

In [ ]:
add_repeating_overlay(
    fig,
    "🌙 Clinic Closed",
    first_start_frame=int((60 * 12) / 10), # After 12 hours, but we only have a frame every 10 minutes
    on_duration_frames=int((60 * 12) / 10),
    off_duration_frames=int((60 * 12) / 10),
    rect_color="navy",
    rect_opacity=0.1
    )

## Text overlay in a different position with no overall overlay 

Let's first generate the basic fig again.

In [ ]:
fig = generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['patient', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=g(),
        entity_col_name="patient",
        debug_mode=True,
        setup_mode=False,
        include_play_button=True,
        start_time="08:00:00",
        entity_icon_size=20,
        resource_icon_size=20,
        gap_between_resource_rows=30,
        plotly_height=700,
        frame_duration=800,
        frame_transition_duration=200,
        plotly_width=1200,
        override_x_max=300,
        override_y_max=500,
        time_display_units="day_clock_ampm",
        display_stage_labels=False,
        add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_1_simplest_case/Simplest%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
    )

fig

In [ ]:
add_repeating_overlay(
    fig,
    "🌙<br>Clinic<br>Closed",
    first_start_frame=int((60 * 12) / 10), # After 12 hours, but we only have a frame every 10 minutes
    on_duration_frames=int((60 * 12) / 10),
    off_duration_frames=int((60 * 12) / 10),
    rect_opacity=0,
    relative_text_position_x=0.85,
    relative_text_position_y=0.7,
    text_font_color="black"
    )